# análise exploratória tabela de movimentações

## Setups

### Definição de constantes

In [1]:
GCP_PROJECT_ID = "pure-league-482018-a7"
DATASET_NAME = "basedosdados"
SCHEMA_NAME = "br_me_caged"
TABLE= "microdados_movimentacao"

### Configurações de ambiente

In [2]:
from google.cloud import bigquery
from pathlib import Path
import os
import warnings

actual_path = Path().absolute()
os.chdir(actual_path.parent.parent.parent)

warnings.filterwarnings("ignore", category=UserWarning, module="google.cloud.bigquery")
client = bigquery.Client(project=GCP_PROJECT_ID)

## análise de metadados

In [3]:
query_metadata_t1 = f"""
    SELECT *
    FROM `{DATASET_NAME}.{SCHEMA_NAME}.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name = '{TABLE}';
"""

df_metadata_t1 = client.query(query_metadata_t1).to_dataframe()
df_metadata_t1

,table_catalog,table_schema,table_name,column_name,ordinal_position,is_nullable,data_type,is_generated,generation_expression,is_stored,...,is_system_defined,is_partitioning_column,clustering_ordinal_position,collation_name,column_default,rounding_mode,data_policies,data_governance_tags,policy_tags,async_generation_status
0,basedosdados,br_me_caged,microdados_movimentacao,ano,1,YES,INT64,NEVER,NaN,NaN,...,NO,YES,<NA>,NULL,NULL,NaN,[],[],[],None
1,basedosdados,br_me_caged,microdados_movimentacao,mes,2,YES,INT64,NEVER,NaN,NaN,...,NO,NO,1,NULL,NULL,NaN,[],[],[],None
2,basedosdados,br_me_caged,microdados_movimentacao,sigla_uf,3,YES,STRING,NEVER,NaN,NaN,...,NO,NO,2,NULL,NULL,NaN,[],[],[],None
3,basedosdados,br_me_caged,microdados_movimentacao,id_municipio,4,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
4,basedosdados,br_me_caged,microdados_movimentacao,cnae_2_secao,5,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
5,basedosdados,br_me_caged,microdados_movimentacao,cnae_2_subclasse,6,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
6,basedosdados,br_me_caged,microdados_movimentacao,saldo_movimentacao,7,YES,INT64,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
7,basedosdados,br_me_caged,microdados_movimentacao,cbo_2002,8,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
8,basedosdados,br_me_caged,microdados_movimentacao,categoria,9,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None
9,basedosdados,br_me_caged,microdados_movimentacao,grau_instrucao,10,YES,STRING,NEVER,NaN,NaN,...,NO,NO,<NA>,NULL,NULL,NaN,[],[],[],None


## Análise de preenchimento

In [4]:
count_list = [f"COUNTIF({col} IS NOT NULL) / COUNT(*) AS {col}_filled_pct" for col in df_metadata_t1['column_name'].values]
count_list_str = ",\n    ".join(count_list)

query_missingness = f"""
SELECT
    {count_list_str}
FROM 
`basedosdados.{SCHEMA_NAME}.{TABLE}`
"""

df2 = client.query(query_missingness).to_dataframe()
df2_long = df2.T.reset_index()
df2_long.columns = ["field", "value"]
df2_long.sort_values(by="value", ascending=False)

,field,value
0,ano_filled_pct,1.000000
1,mes_filled_pct,1.000000
2,sigla_uf_filled_pct,1.000000
4,cnae_2_secao_filled_pct,1.000000
5,cnae_2_subclasse_filled_pct,1.000000
7,cbo_2002_filled_pct,1.000000
6,saldo_movimentacao_filled_pct,1.000000
8,categoria_filled_pct,1.000000
9,grau_instrucao_filled_pct,1.000000
23,origem_informacao_filled_pct,1.000000


## análise de distribuição de valores

### Valores Categoricos

In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import webbrowser


def get_value_counts(client, table, col):
    query = f"""
    SELECT
        `{col}` AS value,
        COUNT(*) AS freq
    FROM `{table}`
    GROUP BY `{col}`
    ORDER BY freq DESC
    
    """
    df = client.query(query).to_dataframe()
    df["value"] = df["value"].astype(str)  # garante eixo categórico consistente
    return df

dtypes=list(df_metadata_t1['data_type'].values)
columns = list(df_metadata_t1['column_name'].values)

cat_columns=[
    col for col, dtype in zip(columns, dtypes) 
    if dtype in ['STRING',"INT64", 'DATE', 'DATETIME', 'TIMESTAMP']
    ]

table = f"{DATASET_NAME}.{SCHEMA_NAME}.{TABLE}"

# Coleta os value_counts de cada coluna (uma query por coluna)
value_counts_dict = {}
for col in cat_columns:
    print(f"Consultando: {col}")
    value_counts_dict[col] = get_value_counts(client, table, col)

# Monta o grid vertical: 1 gráfico por linha
n = len(cat_columns)
fig = make_subplots(
    rows=n, cols=1,
    subplot_titles=[f"Distribuição: {col}" for col in cat_columns],
    vertical_spacing=0.4 / n  # espaçamento proporcional para não sobrepor títulos
)

for i, col in enumerate(cat_columns, start=1):
    df_vc = value_counts_dict[col]
    fig.add_trace(
        go.Bar(
            x=df_vc["value"],
            y=df_vc["freq"],
            name=col,
            showlegend=False
        ),
        row=i, col=1
    )
    fig.update_xaxes(tickangle=45, row=i, col=1)

# Altura por gráfico ~350px, permitindo scroll vertical no notebook/HTML
fig.update_layout(
    height=350 * n,
    width=1000,
    title_text=f"Distribuição de frequência por variável da tabela '{TABLE}'",
    showlegend=False
)

# Salva como HTML
output_path = os.path.abspath(f"data/outputs/distribuicao_{TABLE}_cat_columns.html")
fig.show()

Consultando: ano
Consultando: mes
Consultando: sigla_uf
Consultando: id_municipio
Consultando: cnae_2_secao
Consultando: cnae_2_subclasse
Consultando: saldo_movimentacao
Consultando: cbo_2002
Consultando: categoria
Consultando: grau_instrucao
Consultando: idade
Consultando: raca_cor
Consultando: sexo
Consultando: tipo_empregador
Consultando: tipo_estabelecimento
Consultando: tipo_movimentacao
Consultando: tipo_deficiencia
Consultando: indicador_trabalho_intermitente
Consultando: indicador_trabalho_parcial
Consultando: tamanho_estabelecimento_janeiro
Consultando: indicador_aprendiz
Consultando: origem_informacao
Consultando: indicador_fora_prazo


### Análise colunas numericas

In [6]:
from scripts.num_dist_vizs import *

### Histograma

In [11]:

N_STDDEV_RANGE = 4
MAX_TICKS = 50
N_BINS = 100  # numero de bins dentro da janela exibida (média ± N_STDDEV_RANGE*σ)
 
dtypes = list(df_metadata_t1['data_type'].values)
columns = list(df_metadata_t1['column_name'].values)
 
num_columns = [
    col for col, dtype in zip(columns, dtypes)
    if dtype in ['FLOAT64']
]
 
table = f"{DATASET_NAME}.{SCHEMA_NAME}.{TABLE}"
 
# Coleta estatísticas e histogramas de cada coluna numérica
histogram_dict, stats_dict, valid_columns, bin_width_dict = collect_stats_and_histograms(
    client, table, num_columns, n_bins=N_BINS
)
 
# Monta o gráfico
fig, valid_columns = build_histogram_figure(
    histogram_dict, stats_dict, bin_width_dict, TABLE,
    n_bins=N_BINS, max_ticks=MAX_TICKS
)

fig.show()

Consultando estatísticas: horas_contratuais
Consultando histograma: horas_contratuais (bin_width=0.3144, janela=[16.56, 48])
Consultando estatísticas: salario_mensal
Consultando histograma: salario_mensal (bin_width=95.14, janela=[0, 9514])


### Distribuição contínua

In [12]:

N_STDDEV_RANGE = 4
MAX_TICKS = 50
N_BINS = 100000  # numero de bins dentro da janela exibida (média ± N_STDDEV_RANGE*σ)
 
dtypes = list(df_metadata_t1['data_type'].values)
columns = list(df_metadata_t1['column_name'].values)
 
num_columns = [
    col for col, dtype in zip(columns, dtypes)
    if dtype in ['FLOAT64']
]
 
table = f"{DATASET_NAME}.{SCHEMA_NAME}.{TABLE}"
 
# Coleta estatísticas e histogramas de cada coluna numérica
bins_dict, stats_dict, window_dict, valid_columns = collect_stats_and_bins(
    client, table, num_columns, n_bins=200
)

fig, valid_columns = build_density_figure(
    bins_dict, stats_dict, window_dict, TABLE, log_vals=True
)
 
# Salva como HTML e abre no navegador
output_path = os.path.abspath(f"data/outputs/html/distribuicao_{TABLE}_num_columns.html")
fig.show()
 

### Box Plot

In [9]:
stats_dict, valid_columns = collect_boxplot_stats(client, table, num_columns)

fig, valid_columns = build_boxplot_figure(stats_dict, TABLE, log_vals=False)
output_path = os.path.abspath(f"data/outputs/html/distribuicao_boxplot_{TABLE}_num_columns.html")
fig.show()
 

Consultando quartis: horas_contratuais
Consultando quartis: salario_mensal
